In [1]:
from pyspark import SparkContext, SparkConf
from datetime import datetime


# Initialize SparkContext
conf = SparkConf().setAppName("RDD_CSV_Example").setMaster("local[*]")
sc = SparkContext(conf=conf)

raw_stations_rdd = sc.textFile("data/completeData/stations.csv")
raw_register_rdd = sc.textFile("data/completeData/register.csv")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/24 12:17:50 WARN Utils: Your hostname, cubone, resolves to a loopback address: 127.0.1.1; using 192.168.1.113 instead (on interface wlan0)
26/04/24 12:17:50 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/24 12:17:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
raw_stations_rdd.take(10)

['id\tlongitude\tlatitude\tname',
 '1\t2.180019\t41.397978\tGran Via Corts Catalanes',
 '2\t2.176414\t41.394381\tPlaza TetuÃ¡n',
 '3\t2.181164\t41.393750\tAli Bei',
 '4\t2.181400\t41.393364\tRibes',
 '5\t2.180214\t41.391072\tPg LluÃ\xads Companys',
 '6\t2.180508\t41.391272\tPg LluÃ\xads Companys',
 '7\t2.183183\t41.388867\tPg LluÃ\xads Companys',
 '8\t2.183453\t41.389044\tPasseig lluÃ\xads companys',
 "9\t2.185294\t41.385006\tMarquÃ¨s de l\\'Argentera"]

In [3]:
# Debug
raw_register_rdd.take(10)

['station\ttimestamp\tused_slots\tfree_slots',
 '1\t2008-05-15 12:01:00\t0\t18',
 '1\t2008-05-15 12:02:00\t0\t18',
 '1\t2008-05-15 12:04:00\t0\t18',
 '1\t2008-05-15 12:06:00\t0\t18',
 '1\t2008-05-15 12:08:00\t0\t18',
 '1\t2008-05-15 12:10:00\t0\t18',
 '1\t2008-05-15 12:12:00\t0\t18',
 '1\t2008-05-15 12:14:00\t0\t18',
 '1\t2008-05-15 12:16:00\t0\t18']

In [4]:
# First filter data removing header and wrong entries
def isInvalidReading(line):
    entries = line.split('\t')
    
    used_slots = entries[2]
    free_slots = entries[3]
    
    if used_slots == 0 and free_slots == 0:
        return True
    else:
        return False

header = raw_register_rdd.first()
clean_register_rdd = raw_register_rdd \
    .filter(lambda line: line.strip() != header.strip()) \
    .filter(lambda line: not isInvalidReading(line))

In [5]:
# Map each entry to explicit day of the week and time slot --> Tj
def get_weekday_hour(timestamp):
    datetimeObject = datetime.strptime(timestamp, "%Y-%m-%d %H:%M:%S")
    dayOfTheWeek = datetimeObject.strftime("%a") # Abbreviated weekeday, e.g. Thu for Thusday
    hour = datetimeObject.hour # anything between 12:00 and 12:59 will become 12
    return dayOfTheWeek, hour # types returned is str, int

def get_ST_v_pairs(line):
    entries = line.split('\t')
    station_id = entries[0]
    
    timestamp = entries[1]
    weekday, hour = get_weekday_hour(timestamp)
    
    used_slots = entries[2]
    free_slots = entries[3]

    # Keep station ID as str as it doesn't have semantical meaning as a number
    return (station_id, weekday, hour), (int(used_slots), int(free_slots)) # Use the tuple itself as the key

ST_register_rdd = clean_register_rdd \
    .map(lambda line:  get_ST_v_pairs(line))

In [6]:
# Debug
ST_register_rdd.take(10)

[(('1', 'Thu', 12), (0, 18)),
 (('1', 'Thu', 12), (0, 18)),
 (('1', 'Thu', 12), (0, 18)),
 (('1', 'Thu', 12), (0, 18)),
 (('1', 'Thu', 12), (0, 18)),
 (('1', 'Thu', 12), (0, 18)),
 (('1', 'Thu', 12), (0, 18)),
 (('1', 'Thu', 12), (0, 18)),
 (('1', 'Thu', 12), (0, 18)),
 (('1', 'Thu', 12), (0, 18))]

In [7]:
# Aggregate by key Sj and Tj:
#     read the RDD and count total number of readings,
#     read the RDD and count readings wiht free slots == 0 (pair[1][1])
#     concatenate the two and get the ratio
#
# NOTE: I could cache ST_register_rdd to avoid duplicate readings

# Option1: using CountByKey()
# NOTE: Using CountByKey() I will get a dictionary (it is an action), not a RDD!!!

# Count all readings per key
# tot_readings_per_ST_key = ST_register_rdd.countByKey()

# Count only those with no freeslots through filter()
# tot_critical_readings_per_ST_key = ST_register_rdd.filter(lambda pair: pair[1][1]).countByKey()

In [8]:
# Option 2: Avoid send data to the driver and make a RDD directly

# Count all readings per key
# tot_readings_per_ST_key = ST_register_rdd \
#         .mapValues(lambda v: (v[0], v[1], 1)) \
#         .reduceByKey(lambda v1, v2: (v1[0] + v2[0], v1[1] + v2[1], v1[2] + v2[2])) \
#         .mapValues(lambda v: v[2])

# Count only those with no freeslots through filter()
# tot_critical_readings_per_ST_key = ST_register_rdd \
#     .filter(lambda pair: pair[1][1] == 0) \
#     .mapValues(lambda v: (v[0], v[1], 1)) \
#     .reduceByKey(lambda v1, v2: (v1[0] + v2[0], v1[1] + v2[1], v1[2] + v2[2])) \
#     .mapValues(lambda v: v[2])

In [9]:
# OR ... even better...

# Just do everything in one shot (I dont even need to cache the RDD!):

threshold = 0.2
threshold_bc = sc.broadcast(threshold)

def augment_slots_with_atomical_counters(slots):

    used_slots = slots[0]
    free_slots = slots[1]

    global_counter = 1
    critical_level_free_slots_counter = 1 if free_slots == 0 else 0

    return (global_counter, critical_level_free_slots_counter)

def cumulate_critical_slots_counters(slotsA, slotsB):
    global_counterA = slotsA[0]
    critical_level_free_slots_counterA = slotsA[1]

    global_counterB = slotsB[0]
    critical_level_free_slots_counterB = slotsB[1]

    return (global_counterA + global_counterB, critical_level_free_slots_counterA + critical_level_free_slots_counterB)

crit_readings_per_ST_key = ST_register_rdd \
        .mapValues(augment_slots_with_atomical_counters) \
        .reduceByKey(lambda slotsA, slotsB: cumulate_critical_slots_counters(slotsA, slotsB)) \
        .mapValues(lambda counters: counters[1] / counters[0]) \
        .filter(lambda pair: pair[1] > threshold_bc.value)

In [10]:
# Debug
crit_readings_per_ST_key.count()

2867

In [11]:
# Debug
crit_readings_per_ST_key.take(10)

[(('1', 'Sat', 7), 0.2504472271914132),
 (('1', 'Tue', 3), 0.25277777777777777),
 (('4', 'Sat', 6), 0.2460456942003515),
 (('6', 'Sat', 0), 0.22564102564102564),
 (('7', 'Fri', 5), 0.20469798657718122),
 (('8', 'Fri', 18), 0.23154362416107382),
 (('8', 'Sun', 2), 0.3472222222222222),
 (('9', 'Thu', 21), 0.273972602739726),
 (('9', 'Fri', 16), 0.3055091819699499),
 (('9', 'Sat', 21), 0.3251318101933216)]

In [12]:
def tie_breaker(pair):
    return pair[1], pair[0][2], pair[0][1]

max_critical = crit_readings_per_ST_key.max(key=tie_breaker)
top_five = crit_readings_per_ST_key.takeOrdered(5, key=lambda x: (-x[1], x[0]))

In [13]:
# Debug
max_critical
top_five

[(('221', 'Sun', 23), 0.7095070422535211),
 (('221', 'Sun', 16), 0.6807017543859649),
 (('221', 'Sun', 19), 0.6789473684210526),
 (('221', 'Sun', 17), 0.6771929824561403),
 (('221', 'Sun', 21), 0.6771929824561403)]

In [14]:
# Move S and T to values, and use station id as key
def reorganize_stationwise(kv_pair):
    key = kv_pair[0]
    value = kv_pair[1]

    stationID, weekday, hour = key
    crit_ratio = value

    new_key = stationID
    new_values = (weekday, hour, crit_ratio)
    
    return (new_key, new_values)

crit_readings_per_station = crit_readings_per_ST_key.map(reorganize_stationwise)

In [15]:
# Debug
crit_readings_per_station.take(10)

[('1', ('Sat', 7, 0.2504472271914132)),
 ('1', ('Tue', 3, 0.25277777777777777)),
 ('4', ('Sat', 6, 0.2460456942003515)),
 ('6', ('Sat', 0, 0.22564102564102564)),
 ('7', ('Fri', 5, 0.20469798657718122)),
 ('8', ('Fri', 18, 0.23154362416107382)),
 ('8', ('Sun', 2, 0.3472222222222222)),
 ('9', ('Thu', 21, 0.273972602739726)),
 ('9', ('Fri', 16, 0.3055091819699499)),
 ('9', ('Sat', 21, 0.3251318101933216))]

In [16]:
# This is both associative and commutative, as it is a proxy of a max
def lexicographic_first_weekday_slot(slotA, slotB):
    weekdayA, hourA, crit_ratioA = slotA
    weekdayB, hourB, crit_ratioB = slotB

    return slotA if weekdayA > weekdayB else slotB

# This is both associative and commutative, as it is a proxy of a max
def most_early_slot(slotA, slotB):
    weekdayA, hourA, crit_ratioA = slotA
    weekdayB, hourB, crit_ratioB = slotB

    if hourA == hourB:
        return lexicographic_first_weekday_slot(slotA, slotB)
    else:
        return slotA if hourA > hourB else slotB

# This is both associative and commutative, as it is a proxy of a max
def most_critical_slot(slotA, slotB):
    weekdayA, hourA, crit_ratioA = slotA
    weekdayB, hourB, crit_ratioB = slotB

    if crit_ratioA == crit_ratioB:
        return most_early_slot(slotA, slotB)
    else:
        return slotA if crit_ratioA >= crit_ratioB else slotB

# Take the max per each stattion
most_crit_readings_per_station = crit_readings_per_station.reduceByKey(most_critical_slot)

In [17]:
# Debug
most_crit_readings_per_station.take(10)

[('7', ('Fri', 22, 0.37520938023450584)),
 ('114', ('Mon', 0, 0.39166666666666666)),
 ('131', ('Mon', 0, 0.25555555555555554)),
 ('136', ('Sat', 5, 0.28421052631578947)),
 ('130', ('Sun', 5, 0.21441124780316345)),
 ('8', ('Sun', 0, 0.41388888888888886)),
 ('20', ('Tue', 1, 0.3888888888888889)),
 ('25', ('Tue', 9, 0.47)),
 ('149', ('Wed', 9, 0.2824561403508772)),
 ('171', ('Wed', 17, 0.25749559082892415))]

In [18]:
# Stop the context
sc.stop()